# Tata Technologies Ltd. - TechPulse FY-26: Applied AI & ML
## Lab Statement 2: Simulated Driving Agent Behavior

**Curriculum Context:** Unit 1 – Introduction to AI & ML (Types of ML, Evolutionary Algorithms & Autonomous Agents)  
**Track:** AI & ML | **Level:** Intermediate  
**Core Technology:** Python, Genetic Algorithms (GA), Neural Network Policy Controllers, Kinematic Vehicle Simulation  
**Industrial Application:** Autonomous Vehicle Path Planning, Adaptive Cruise Control & Collision Avoidance Optimization  

---

### 🎯 Learning Objectives
By completing this laboratory assignment, students will be able to:
1. Formulate self-driving vehicle control as an evolutionary search optimization problem.
2. Model kinematic vehicle state dynamics (heading, velocity, steering) and multi-directional raycast distance sensors (virtual LIDAR).
3. Encode neural network policy controller weights into fixed-length floating-point chromosomes.
4. Design multi-objective fitness functions balancing forward distance progress, checkpoint navigation, and collision avoidance.
5. Implement genetic operators: **Tournament Selection**, **Arithmetic Crossover**, **Gaussian Mutation**, and **Elitism**.
6. Analyze generational convergence, lap completion rates, and trajectory optimization curves.

---

### Step 0: Imports & Environment Configuration

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
print("Environment initialized successfully!")

### Step 1: Autonomous Vehicle Kinematics & Raycast Sensor Mathematics

The vehicle state vector at time step $t$ is defined as $\mathbf{s}_t = [x_t, y_t, \theta_t, v_t]^T$, where:
- $(x_t, y_t)$: 2D Cartesian spatial coordinates
- $\theta_t$: Heading orientation angle
- $v_t$: Linear forward velocity

The state transitions obey non-holonomic kinematic bicycle approximations:
$$\theta_{t+1} = \theta_t + \delta_t \cdot \delta_{\max} \cdot \left(\frac{v_t}{v_{\text{ref}}}\right) \cdot \Delta t$$
$$x_{t+1} = x_t + v_t \cos(\theta_{t+1}) \Delta t$$
$$y_{t+1} = y_t + v_t \sin(\theta_{t+1}) \Delta t$$

The agent senses the track boundaries using **5 virtual LIDAR raycasts** oriented at relative angles $\phi \in \{-60^\circ, -30^\circ, 0^\circ, +30^\circ, +60^\circ\}$ relative to the vehicle's forward heading vector.

### Step 2: Track Generation & Geometric Boundary Construction

In [ ]:
t = np.linspace(0, 2 * np.pi, 100)
center_x = 120.0 * np.cos(t) + 30.0 * np.cos(3 * t)
center_y = 70.0 * np.sin(t) - 20.0 * np.sin(2 * t)

track_width = 16.0
half_w = track_width / 2.0

dx = np.gradient(center_x)
dy = np.gradient(center_y)
lengths = np.sqrt(dx**2 + dy**2)
nx = -dy / lengths
ny = dx / lengths

inner_x, inner_y = center_x - half_w * nx, center_y - half_w * ny
outer_x, outer_y = center_x + half_w * nx, center_y + half_w * ny

centerline = np.column_stack([center_x, center_y])
inner_bound = np.column_stack([inner_x, inner_y])
outer_bound = np.column_stack([outer_x, outer_y])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(centerline[:, 0], centerline[:, 1], 'k--', alpha=0.4, label='Centerline')
ax.plot(inner_bound[:, 0], inner_bound[:, 1], 'dimgray', linewidth=2, label='Inner Wall')
ax.plot(outer_bound[:, 0], outer_bound[:, 1], 'dimgray', linewidth=2, label='Outer Wall')
ax.fill(np.append(outer_bound[:, 0], inner_bound[::-1, 0]), np.append(outer_bound[:, 1], inner_bound[::-1, 1]), color='#ecf0f1', alpha=0.6)
ax.set_title('Race Track Geometry & Boundary Walls')
ax.set_aspect('equal')
ax.legend()
plt.show()

### Step 3: Neural Network Driving Policy & Genetic Chromosome Encoding

Each driving agent's brain is a compact multi-layer feedforward neural network:
- **Input Layer (6 features)**: 5 normalized LIDAR range sensor readings $[d_{-60}, d_{-30}, d_0, d_{+30}, d_{+60}] \in [0, 1]$ and normalized velocity $\frac{v}{6.0}$.
- **Hidden Layer (8 neurons)**: Hyperbolic tangent activation $\tanh(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1)$.
- **Output Layer (2 actuators)**:
  - Steering Angle: $\delta = \tanh(z_1) \in [-1, 1]$ (scaled to $\pm 32^\circ$)
  - Throttle / Acceleration: $a = \sigma(z_2) = \frac{1}{1 + e^{-z_2}} \in [0, 1]$

**Chromosome Representation:** Vector of 74 continuous real values:
$$\text{Chromosome Size} = (6 \times 8 + 8) + (8 \times 2 + 2) = 56 + 18 = 74\text{ genes}$$

### Step 4: Multi-Objective Fitness Function & Genetic Operators

1. **Fitness Evaluation**:
$$f(\text{agent}) = 100 \cdot N_{\text{checkpoints}} + 1.5 \cdot d_{\text{total}} + 0.2 \cdot t_{\text{steps}}$$
- Checkpoints passed rewards directional circuit progress.
- Distance traveled rewards smooth forward trajectory.
- Collision immediately terminates simulation for that agent.

2. **Selection**: **Tournament Selection** with size $k=3$ to maintain genetic diversity and selective pressure.
3. **Crossover**: **Arithmetic Blending Crossover** with $\alpha \sim U(0.2, 0.8)$:
$$\mathbf{c}_1 = \alpha \mathbf{p}_1 + (1 - \alpha) \mathbf{p}_2, \quad \mathbf{c}_2 = (1 - \alpha) \mathbf{p}_1 + \alpha \mathbf{p}_2$$
4. **Mutation**: **Adaptive Gaussian Perturbation**:
$$c_i \leftarrow c_i + \mathcal{N}(0, \sigma^2), \quad \text{with mutation probability } p_m = 0.08$$
5. **Elitism**: Top 2 performing agents are copied directly into the next generation without modification.

### Step 5: Simulation Telemetry & Generational Convergence Results
Loading and visualizing the recorded generational telemetry from `simulation_logs.csv`.

In [ ]:
logs_df = pd.read_csv('simulation_logs.csv')
print("Evolutionary Simulation Telemetry Log:")
display(logs_df.head(10))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(logs_df['generation'], logs_df['max_fitness'], color='#27ae60', linewidth=2.5, marker='o', label='Peak Fitness (Best Agent)')
ax.plot(logs_df['generation'], logs_df['mean_fitness'], color='#2980b9', linewidth=2.0, linestyle='--', label='Population Mean Fitness')
ax.fill_between(logs_df['generation'], logs_df['mean_fitness'], logs_df['max_fitness'], color='#2ecc71', alpha=0.15)
ax.set_title('Genetic Algorithm Convergence & Fitness Progression', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Generation Number')
ax.set_ylabel('Fitness Score')
ax.legend()
plt.tight_layout()
plt.show()

### Step 6: Visualizing Trajectory Evolution Across Generations

In [ ]:
from IPython.display import Image
Image('plots/01_track_and_best_agent_trajectories.png')

### Step 7: Lap Progress & Distance per Generation

In [ ]:
Image('plots/03_lap_completion_and_distance.png')

### Step 8: Industrial Takeaways & Autonomous Driving Insights

1. **Rapid Initial Convergence**: Early generations (Gen 1–5) quickly discover basic wall-avoidance behavior, dramatically increasing mean fitness.
2. **Overcoming Challenging Hairpin Curves**: Complex high-curvature sectors require coordinating forward speed reduction with steering angle; arithmetic crossover and fine Gaussian mutation enable the population to navigate tight corners without colliding.
3. **Elitism Prevents Regression**: Top-2 elitism ensures that successful driving strategies discovered in previous generations are never lost to destructive mutations.
4. **Real-World Parallels**: This evolutionary approach mirrors real-world automotive autonomous testing (e.g. neuroevolution and reinforcement learning calibration in simulation environments before vehicle hardware deployment).